# 3.9 NumPy Fundamentals

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 3: Python for Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-03-python-for-machine-learning/03.09_numpy-fundamentals.ipynb)

Companion code for Section 3.9 of the book. Run the cells in order from top to bottom.

**Example 3.9.1**

In [ ]:
import numpy as np

# From Python lists
a = np.array([1, 2, 3, 4, 5])
b = np.array([[1,2,3],[4,5,6],[7,8,9]])   # 2D array (matrix)
c = np.array([1, 2, 3], dtype=np.float32) # Explicit dtype

print(f'a: shape={a.shape}, dtype={a.dtype}')   # (5,), int64
print(f'b: shape={b.shape}, dtype={b.dtype}')   # (3,3), int64
print(f'c: shape={c.shape}, dtype={c.dtype}')   # (3,), float32

# Factory functions - most common in ML
print('\nFactory functions:')
print(np.zeros((3, 4)))             # 3×4 matrix of zeros - initialise weights
print(np.ones((2, 3)))              # 2×3 matrix of ones
print(np.eye(4))                    # 4×4 identity matrix
print(np.full((2, 3), 7.0))        # Fill with a constant value

# Ranges and sequences
print(np.arange(0, 10, 2))         # [0 2 4 6 8] - like Python range
print(np.linspace(0, 1, 6))        # [0. 0.2 0.4 0.6 0.8 1.] - n evenly spaced
print(np.logspace(0, 3, 4))        # [1. 10. 100. 1000.] - log scale

# Random arrays
np.random.seed(42)
print(np.random.randn(3, 4))       # Standard normal N(0,1)
print(np.random.rand(2, 3))        # Uniform [0, 1)
print(np.random.randint(0, 10, (3, 3)))  # Random integers
print(np.random.choice([1,2,3,4,5], size=10, replace=True))

# Important dtypes for ML
print('\nDtype memory usage:')
for dtype in [np.int8, np.int32, np.int64, np.float16, np.float32, np.float64]:
    arr = np.zeros(1, dtype=dtype)
    print(f'  {str(dtype.__name__):10s}: {arr.itemsize} bytes per element')

**Example 3.9.2**

In [ ]:
import numpy as np

# Dataset simulation: 1000 samples × 5 features
np.random.seed(0)
X = np.random.randn(1000, 5)
y = np.random.randint(0, 3, 1000)   # 3 classes

# Basic indexing
print('First sample:      ', X[0])          # Row 0 (all features)
print('Feature 2 only:    ', X[0, 2])       # Row 0, col 2
print('Last 3 samples:    ', X[-3:].shape)  # (3, 5)
print('Cols 1-3:          ', X[:, 1:4].shape) # All rows, cols 1,2,3 → (1000,3)
print('Corner block:      ', X[:3, :3])     # Top-left 3×3 block

# Boolean (mask) indexing - essential for data filtering
positive_class = X[y == 2]          # Only samples from class 2
print(f'\nClass 2 samples: {positive_class.shape}')

large_features = X[X[:,0] > 2.0]   # Rows where feature 0 > 2.0
print(f'High feature-0 samples: {large_features.shape}')

# Fancy indexing - select arbitrary rows by index array
batch_idx = np.array([5, 23, 100, 450, 999])
batch     = X[batch_idx]
print(f'Custom batch: {batch.shape}')        # (5, 5)

# Train/val/test split using indexing
n = len(X)
idx = np.random.permutation(n)              # Shuffle indices
train_end = int(0.7 * n)
val_end   = int(0.85 * n)

X_train = X[idx[:train_end]]
X_val   = X[idx[train_end:val_end]]
X_test  = X[idx[val_end:]]
y_train = y[idx[:train_end]]
y_val   = y[idx[train_end:val_end]]
y_test  = y[idx[val_end:]]

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

**Example 3.9.3**

In [ ]:
import numpy as np

# Element-wise operations - vectorised, no loops needed
a = np.array([1., 4., 9., 16., 25.])
print('sqrt:', np.sqrt(a))         # [1. 2. 3. 4. 5.]
print('log:', np.log(a))           # Natural log
print('exp:', np.exp([0,1,2]))     # [1. e e²]
print('abs:', np.abs([-1,-2,3]))   # [1 2 3]

# Reduction operations
X = np.random.randn(100, 10)       # 100 samples, 10 features
print(f'Global mean:           {X.mean():.4f}')
print(f'Mean per feature:    {X.mean(axis=0).shape}')   # (10,)
print(f'Mean per sample:    {X.mean(axis=1).shape}')   # (100,)
print(f'Feature std:         {X.std(axis=0).round(3)}')
print(f'Min/Max per feature: {X.min(axis=0).round(2)}')

# Aggregation with axis
# axis=0: collapse rows (operate down columns) → result has shape of one row
# axis=1: collapse columns (operate across rows) → result has shape of one column
print(f'Sum all:        {X.sum():.2f}')
print(f'Sum per column: {X.sum(axis=0).shape}')  # (10,)
print(f'Sum per row:    {X.sum(axis=1).shape}')  # (100,)

# Broadcasting - automatic shape matching
# Standardise features: (X - mean) / std - the most common ML operation
mean = X.mean(axis=0)   # Shape (10,)
std  = X.std(axis=0)    # Shape (10,)
X_scaled = (X - mean) / std   # Broadcasting: (100,10) - (10,) → (100,10)
print(f'\nAfter scaling:')
print(f'  Mean: {X_scaled.mean(axis=0).round(4)}')    # Should be ~0
print(f'  Std:  {X_scaled.std(axis=0).round(4)}')     # Should be ~1

# Vectorised operations vs loop - performance comparison
import time
n = 1_000_000
a = np.random.randn(n)
b = np.random.randn(n)

# Python loop
start = time.time()
result_loop = [x*y for x,y in zip(a,b)]
loop_time = time.time() - start

# NumPy vectorised
start = time.time()
result_np = a * b
np_time = time.time() - start

print(f'\nFor 1M element multiply:')
print(f'  Python loop: {loop_time*1000:.1f}ms')
print(f'  NumPy:       {np_time*1000:.2f}ms')
print(f'  Speedup:     {loop_time/np_time:.0f}x')

**Example 3.9.4**

In [ ]:
import numpy as np

# Reshape - change shape without copying data
a = np.arange(24)
print('Original:', a.shape)            # (24,)
b = a.reshape(4, 6)                    # 4 rows, 6 cols
c = a.reshape(2, 3, 4)                 # 3D tensor
d = a.reshape(-1, 6)                   # -1: NumPy infers this dimension (=4)
print('Reshaped (4,6):', b.shape)      # (4, 6)
print('Reshaped (2,3,4):', c.shape)    # (2, 3, 4)
print('Reshaped (-1,6):', d.shape)     # (4, 6)

# Flatten - collapse to 1D
img = np.random.rand(28, 28)           # Grayscale image
flat = img.flatten()                   # → (784,)  [copies data]
flat = img.ravel()                     # → (784,)  [view if possible - faster]
print(f'Image {img.shape} → flat {flat.shape}')

# Transpose and axis manipulation
X = np.random.randn(10, 5)            # 10 samples × 5 features
print(f'X: {X.shape} → Xᵀ: {X.T.shape}')   # (5, 10)

# Stack arrays
a = np.array([[1,2],[3,4]])
b = np.array([[5,6],[7,8]])
print('vstack:', np.vstack([a,b]).shape)   # Stack vertically → (4,2)
print('hstack:', np.hstack([a,b]).shape)   # Stack horizontally → (2,4)
print('dstack:', np.dstack([a,b]).shape)   # Stack depth-wise → (2,2,2)

# Concatenate along any axis
X_train = np.random.randn(800, 5)
X_test  = np.random.randn(200, 5)
X_all   = np.concatenate([X_train, X_test], axis=0)
print(f'Full dataset: {X_all.shape}')      # (1000, 5)

# Add/remove dimensions
v = np.array([1,2,3])                       # Shape (3,)
print(v[np.newaxis, :].shape)               # (1,3) - add row dim
print(v[:, np.newaxis].shape)               # (3,1) - add col dim
print(np.expand_dims(v, axis=0).shape)      # (1,3)
print(np.squeeze(v[np.newaxis]).shape)       # (3,) - remove size-1 dims

# Sorting and argsort
scores = np.array([0.91, 0.84, 0.97, 0.76, 0.93])
sorted_idx = np.argsort(scores)[::-1]       # Descending
print('Rankings:', sorted_idx)              # [2 4 0 1 3]
print('Top 3 scores:', scores[sorted_idx[:3]])

**Example 3.9.5**

In [ ]:
import numpy as np
# Matrix multiplication - the core DL operation
W = np.random.randn(5, 3)    # Weight matrix: 5 inputs → 3 outputs
X = np.random.randn(10, 5)   # Batch of 10 samples, 5 features
b = np.zeros(3)              # Bias
Z = X @ W + b                # Linear layer forward pass
print(f'Layer output: {Z.shape}')   # (10, 3)

# Linear algebra operations
A = np.array([[3.,1.],[1.,2.]])
print('Determinant:    ', np.linalg.det(A))              # 5.0
print('Trace:          ', np.trace(A))                   # 5.0
print('Rank:           ', np.linalg.matrix_rank(A))      # 2
print('Inverse:\n',       np.round(np.linalg.inv(A),3))

# Eigendecomposition
evals, evecs = np.linalg.eig(A)
print(f'Eigenvalues: {np.round(evals,4)}')

# Solving linear systems: Ax = b
A = np.array([[2.,1.,1.],[4.,3.,3.],[8.,7.,9.]])
b = np.array([1., 1., -1.])
x = np.linalg.solve(A, b)   # Solve Ax = b
print(f'Solution x = {np.round(x,4)}')
print(f'Verify Ax  = {np.round(A@x,4)} (should equal b={b})')

# Norms
v = np.array([3., 4., 0.])
print(f'L2 norm: {np.linalg.norm(v):.4f}')          # 5.0
print(f'L1 norm: {np.linalg.norm(v, ord=1):.4f}')   # 7.0

# SVD
M = np.random.randn(4, 3)
U, s, Vt = np.linalg.svd(M, full_matrices=False)
print(f'SVD: U{U.shape} · s{s.shape} · Vt{Vt.shape}')